# CareerBridge AI — fine-tune a Transformer resume classifier (Kaggle GPU)

**Before running:** Settings → Accelerator **GPU T4 x2** (or P100) · Internet **On** · Add Data → your private dataset **careerbridge-resumes-v2**.
Then **Run All**. When finished, download `careerbridge-transformer.zip` from the Output panel (right side).

* Same train/test split as the TF-IDF models (v2/v3), so results are directly comparable.
* 10% of the training data is held out for validation to pick the best epoch; the **test set is used once, at the end**.
* The model is exported to ONNX (int8) so CareerBridge runs it on CPU with onnxruntime — no PyTorch needed there.

In [ ]:
import glob, json, os, time
import numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import Dataset

MODEL_NAME = "microsoft/deberta-v3-small"
MAX_LEN, HEAD, TAIL = 512, 382, 128          # long resumes: keep the start and the end
EPOCHS, LR, BATCH, SEED = 4, 3e-5, 16, 42
OUT = "/kaggle/working/careerbridge-transformer"
os.makedirs(OUT, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable a GPU in Settings!")

In [ ]:
data_dir = os.path.dirname(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
train_df = pd.read_csv(f"{data_dir}/train.csv"); test_df = pd.read_csv(f"{data_dir}/test.csv")
labels = json.load(open(f"{data_dir}/labels.json")); label2id = {l: i for i, l in enumerate(labels)}
train_df["y"] = train_df["label"].map(label2id); test_df["y"] = test_df["label"].map(label2id)
tr_df, val_df = train_test_split(train_df, test_size=0.1, stratify=train_df["y"], random_state=SEED)
print(len(tr_df), "train /", len(val_df), "validation /", len(test_df), "test /", len(labels), "labels")

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(batch):
    ids = tok(batch["text"], add_special_tokens=False, truncation=False)["input_ids"]
    out = []
    for x in ids:
        if len(x) > MAX_LEN - 2:
            x = x[:HEAD] + x[-TAIL:]
        out.append([tok.cls_token_id] + x + [tok.sep_token_id])
    return {"input_ids": out, "attention_mask": [[1] * len(x) for x in out], "labels": batch["y"]}

def to_ds(df):
    return Dataset.from_pandas(df[["text", "y"]].reset_index(drop=True)).map(encode, batched=True, remove_columns=["text", "y"])

ds_train, ds_val, ds_test = to_ds(tr_df), to_ds(val_df), to_ds(test_df)

In [ ]:
def metrics(pred):
    logits, y = pred
    p = logits.argmax(-1)
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, average="macro", zero_division=0)
    top3 = np.mean([t in row for t, row in zip(y, np.argsort(-logits, axis=1)[:, :3])])
    return {"accuracy": accuracy_score(y, p), "precision_macro": pr, "recall_macro": rc, "f1_macro": f1, "top3_accuracy": top3}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(labels), id2label=dict(enumerate(labels)), label2id=label2id)
args = TrainingArguments(
    output_dir="/kaggle/working/run", num_train_epochs=EPOCHS, learning_rate=LR, weight_decay=0.01, warmup_ratio=0.1,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=32, fp16=torch.cuda.is_available(),
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1, load_best_model_at_end=True,
    metric_for_best_model="f1_macro", logging_steps=50, report_to="none", seed=SEED)
trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
start = time.time(); trainer.train(); train_seconds = time.time() - start
print(f"training took {train_seconds/60:.1f} min")

In [ ]:
# Final, one-time evaluation on the held-out test set.
pred = trainer.predict(ds_test)
test_metrics = metrics((pred.predictions, pred.label_ids))
report = classification_report(pred.label_ids, pred.predictions.argmax(-1), target_names=labels, output_dict=True, zero_division=0)
val_metrics = trainer.evaluate()
print(json.dumps({k: round(float(v), 4) for k, v in test_metrics.items()}, indent=2))

In [ ]:
# Export to ONNX, then quantise to int8 for fast CPU inference.
from onnxruntime.quantization import quantize_dynamic, QuantType
model = trainer.model.eval().to("cpu")
dummy = tok("sample resume text", return_tensors="pt")
torch.onnx.export(model, (dummy["input_ids"], dummy["attention_mask"]), f"{OUT}/model_fp32.onnx",
                  input_names=["input_ids", "attention_mask"], output_names=["logits"], opset_version=17,
                  dynamic_axes={"input_ids": {0: "batch", 1: "seq"}, "attention_mask": {0: "batch", 1: "seq"}, "logits": {0: "batch"}})
quantize_dynamic(f"{OUT}/model_fp32.onnx", f"{OUT}/model.onnx", weight_type=QuantType.QInt8)
os.remove(f"{OUT}/model_fp32.onnx")
tok.backend_tokenizer.save(f"{OUT}/tokenizer.json")

# Check the int8 model still agrees with PyTorch on a few test resumes.
import onnxruntime as ort
sess = ort.InferenceSession(f"{OUT}/model.onnx", providers=["CPUExecutionProvider"])
agree = 0
for i in range(50):
    ex = ds_test[i]
    ids = np.array([ex["input_ids"]], dtype=np.int64); mask = np.array([ex["attention_mask"]], dtype=np.int64)
    agree += int(sess.run(None, {"input_ids": ids, "attention_mask": mask})[0].argmax() == pred.predictions[i].argmax())
print(f"ONNX int8 agrees with PyTorch on {agree}/50 test resumes")

In [ ]:
json.dump({"model_name": MODEL_NAME, "labels": labels, "max_len": MAX_LEN, "head": HEAD, "tail": TAIL,
           "cls_id": tok.cls_token_id, "sep_id": tok.sep_token_id, "pad_id": tok.pad_token_id},
          open(f"{OUT}/config.json", "w"), indent=2)
json.dump({"test": {k: float(v) for k, v in test_metrics.items()},
           "validation": {k: float(v) for k, v in val_metrics.items() if isinstance(v, (int, float))},
           "per_class": report, "train_seconds": train_seconds, "epochs": EPOCHS, "learning_rate": LR,
           "onnx_agreement_50": agree, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
          open(f"{OUT}/metrics.json", "w"), indent=2)
import shutil
shutil.make_archive("/kaggle/working/careerbridge-transformer", "zip", OUT)
print("Done → download careerbridge-transformer.zip from the Output panel")
print(sorted(os.listdir(OUT)), round(os.path.getsize(f"{OUT}/model.onnx") / 1e6, 1), "MB")